# 03 - Outliers de distancia

Esta etapa analisa `race_distance` a partir de `dataset_limpo.csv`. Corridas excepcionalmente encurtadas sao removidas por uma lista revisada de `race_id`; a saida e salva em `dataset_limpo_sem_corridas_encurtadas.csv`. O arquivo de entrada permanece preservado.

Como `race_distance` se repete nas linhas dos pilotos, a triagem considera uma linha por corrida.


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 40)
pd.set_option('display.width', 160)

CAMINHO_DADOS = Path('dataset_limpo.csv')
df = pd.read_csv(CAMINHO_DADOS, low_memory=False)
df_historico = df.copy()

print(f'Arquivo: {CAMINHO_DADOS}')
print(f'Dimensoes: {df.shape[0]:,} linhas x {df.shape[1]:,} colunas')
print(f'Corridas: {df["race_id"].nunique():,}')

Arquivo: dataset_limpo.csv
Dimensoes: 9,146 linhas x 137 colunas
Corridas: 436


## 1. Distancia oficial da corrida

`race_distance` aparece repetida em cada linha de piloto. A triagem usa uma linha por `race_id` para que cada corrida tenha o mesmo peso. Monaco fica fora das tabelas e dos graficos: sua distancia menor e uma caracteristica conhecida do circuito e comprimiria a escala desta comparacao.

O IQR destaca distancias incomuns para revisao. Corridas podem ser encurtadas por clima, acidentes, interrupcoes ou problemas na largada, entao um valor sinalizado nao e automaticamente um erro.


In [ ]:
colunas_evento = [
    'race_id', 'race_year', 'race_round', 'gp_id', 'circuit_name',
    'race_distance', 'race_laps'
]
df_corridas = df[colunas_evento].drop_duplicates(subset='race_id').copy()
# Monaco nao entra nas tabelas nem nos graficos desta analise.
df_corridas = df_corridas.loc[df_corridas['gp_id'].ne('monaco')].copy()

# Confere se os valores que deveriam ser por corrida estao consistentes nas linhas de pilotos.
consistencia_corridas = df.groupby('race_id')[
    ['race_distance', 'race_laps']
].nunique(dropna=False)
inconsistencias = consistencia_corridas.gt(1).sum().rename('corridas_com_valores_divergentes')
print('Corridas com valores divergentes dentro do mesmo race_id:')
display(inconsistencias.to_frame())

valores = df_corridas['race_distance'].dropna()
q1, q3 = valores.quantile([0.25, 0.75])
iqr = q3 - q1
limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

mask_iqr = df_corridas['race_distance'].notna() & (
    (df_corridas['race_distance'] < limite_inferior)
    | (df_corridas['race_distance'] > limite_superior)
)
df_corridas['fora_do_iqr'] = mask_iqr

print(f'Corridas analisadas (sem Monaco): {len(valores):,}')
print(f'Q1: {q1:.3f} km | Q3: {q3:.3f} km | IQR: {iqr:.3f} km')
print(f'Limites IQR: {limite_inferior:.3f} a {limite_superior:.3f} km')
print(f'Corridas sinalizadas pelo IQR: {int(mask_iqr.sum())}')
print('Corridas fora dos limites IQR:')
display(df_corridas.loc[mask_iqr].sort_values('race_distance'))

In [ ]:
# Distribuicao antes da remocao: os mesmos limites IQR usados na tabela acima.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={'width_ratios': [2, 1]})
axes[0].hist(valores, bins=25, range=(0, 320), color='#2878B5', edgecolor='white')
axes[0].axvline(limite_inferior, color='#D9534F', linestyle='--', label='Limite IQR')
axes[0].axvline(limite_superior, color='#D9534F', linestyle='--')
axes[0].set(title='Antes da remocao', xlabel='Distancia (km)', ylabel='Numero de corridas', xlim=(0, 320))
axes[0].legend()
axes[1].boxplot(valores, orientation='horizontal', tick_labels=['Corridas'], patch_artist=True,
                boxprops={'facecolor': '#9CC9E8'})
axes[1].axvline(limite_inferior, color='#D9534F', linestyle='--')
axes[1].axvline(limite_superior, color='#D9534F', linestyle='--')
axes[1].set(title='Boxplot IQR - antes', xlabel='Distancia (km)', xlim=(0, 320))
plt.tight_layout()
plt.show()

## Decisao: provas encurtadas removidas

O IQR e uma triagem: ele aponta distancias incomuns, mas nao decide sozinho quais dados excluir. Conferimos os eventos sinalizados e removemos 33 corridas cuja distancia oficial ficou abaixo do padrao por circunstancias excepcionais ou por erros na sinalizacao da bandeira quadriculada. A decisao considera cada corrida inteira: todas as linhas de pilotos associadas ao `race_id` saem juntas, para nao deixar um evento incompleto.

Esses resultados nao sao erros de registro em todos os casos. Sao resultados oficiais de corridas que nao percorreram a distancia prevista, por limite de tempo, voltas de formacao adicionais ou interrupcao de largada. China 2014 e Canada 2018 sao casos diferentes: a bandeira quadriculada foi mostrada antes da hora, e a classificacao oficial considerou uma volta anterior. Para esta etapa, removemos todas porque o objetivo e analisar e preparar corridas com distancia representativa do formato normal.

Monaco fica fora dos dois graficos e das duas tabelas de outliers por ser naturalmente mais curto. Isso e apenas um filtro da analise visual; nao remove as corridas regulares de Monaco do dataset preparado para modelagem. Monaco 2022 continua removido da base filtrada porque a prova terminou apos 64 das 78 voltas previstas, depois de atrasos por chuva e do limite de tempo.

| race_id | corrida | distancia registrada | motivo | fonte |
| ---: | --- | ---: | --- | --- |
| 700 | Brasil 2003 | 232.656 km | Chuva e acidentes levaram a bandeira vermelha antes da distancia prevista. | [F1: historico do GP](https://www.formula1.com/en/latest/article/moments-in-time-the-brazilian-grand-prix.5dr3Egu9lleFLsheSjlkQ0) |
| 805 | Malasia 2009 | 171.833 km | Chuva intensa interrompeu a prova na volta 31 de 56. | [F1: GP da Malasia](https://www.formula1.com/en/latest/article/need-to-know-malaysia.3HZ93OGO7KIQys4GQe0siO) |
| 912 | Japao 2014 | 255.208 km | A prova terminou na volta 44 apos o acidente de Jules Bianchi. | [F1: relato da corrida](https://www.formula1.com/en/latest/article/race-hamilton-wins-rain-affected-race-in-japan.FOQ9DKAeOj9aMaFMy2Ado) |
| 1047 | Belgica 2021 | 6.880 km | Chuva extrema permitiu apenas poucas voltas atras do safety car. | [F1: relato da corrida](https://www.formula1.com/en/latest/article/verstappen-takes-victory-in-severely-shortened-rain-affected-belgian-gp-as.4AqGhiKQfFaqr7KZjyDjPZ) |
| 1064 | Monaco 2022 | 213.568 km | A prova foi limitada pelo tempo e terminou apos 64 das 78 voltas, depois de atrasos por chuva. | [FIA: classificacao](https://www.fia.com/sites/default/files/decision-document/2022%20Monaco%20Grand%20Prix%20-%20Final%20Race%20Classification.pdf); [FIA: relato](https://www.fia.com/news/f1-perez-wins-monaco-ahead-sainz-and-verstappen) |
| 1075 | Japao 2022 | 162.296 km | Chuva e bandeira vermelha; a prova terminou apos 28 das 53 voltas. | [F1: relato da corrida](https://www.formula1.com/en/latest/article/verstappen-crowned-world-champion-with-japanese-gp-victory-after-late.7g1xgGHHew3f2If32u8U1z) |
| 703 | Austria 2003 | 298.494 km | Duas largadas foram abortadas por falha no launch control de Cristiano da Matta; cada procedimento extra retirou uma volta da prova. | [Toyota: relato da equipe](https://toyotagazooracing.com/archive/ms/en/F1archive/season/gp2003/06_austria/03_ast_07.html) |
| 872 | Singapura 2012 | 299.170 km | O limite de duas horas foi atingido antes das 61 voltas previstas; a prova terminou em 59. | [FIA: retrospectiva](https://www.fia.com/news/2013-singapore-grand-prix-preview) |
| 901 | China 2014 | 294.164 km | A bandeira quadriculada foi mostrada uma volta antes; pela regra, a classificacao voltou ao fim da volta 54. | [F1: resultado oficial](https://www.formula1.com/en/results/2014/races/901/china/race-result) |
| 966 | Gra-Bretanha 2017 | 300.307 km | Jolyon Palmer abandonou na volta de formacao; a volta de formacao extra reduziu a prova de 52 para 51 voltas. | [FIA: relato e volta extra](https://www.fia.com/news/f1-hamilton-takes-record-equalling-fifth-british-gp-win-ferraris-hopes-are-punctured); [Honda: distancia reduzida](https://global.honda/en/F1/race/2017/GreatBritain/) |
| 970 | Singapura 2017 | 293.633 km | Corrida molhada com tres periodos de safety car; terminou no limite de duas horas, apos 58 das 61 voltas. | [F1: relato da corrida](https://www.formula1.com/en/latest/article/hamilton-wins-dramatic-wet-dry-grand-prix-in-singapore.22IsGiaxgEUKYUGcCIMQsA.22IsGiaxgEUKYUGcCIMQsA) |
| 983 | Canada 2018 | 296.548 km | A bandeira quadriculada foi mostrada cedo por engano; a classificacao oficial ficou em 68 das 70 voltas. | [FIA: classificacao](https://www.fia.com/events/fia-formula-one-world-championship/season-2018/race-classification-5); [Autosport: explicacao do erro](https://porschecarshistory.com/wp-content/old/lib/magazines/autosport/2018/06/AS2018.06.14.pdf) |
| 1008 | Alemanha 2019 | 292.736 km | Baixa visibilidade pela chuva exigiu tres voltas de formacao atras do safety car, reduzindo a prova de 67 para 64 voltas. | [Honda: resultado e nota da corrida](https://global.honda/en/F1/race/2019/Germany/) |
| 1074 | Singapura 2022 | 298.580 km | A chuva atrasou a largada; a prova chegou ao limite de duas horas apos 59 das 61 voltas. | [FIA: relato da corrida](https://www.fia.com/news/f1-perez-penalised-wins-singapore-ahead-leclerc-verstappen-finishes-seventh) |
| 1093 | Italia 2023 | 295.134 km | A largada foi abortada apos Yuki Tsunoda parar na volta de formacao; as voltas extras reduziram a prova de 53 para 51. | [F1: incidente e largada](https://www.formula1.com/en/latest/article/race-start-watch-the-exciting-start-at-the-italian-grand-prix-as-sainz-keeps.cWjYZhiKDbnjValRPJod8) |
| 1126 | Australia 2025 | 300.846 km | Isack Hadjar bateu na volta de formacao e a largada foi abortada, reduzindo a prova de 58 para 57 voltas. | [F1: relato da corrida](https://www.formula1.com/en/latest/article/norris-beats-verstappen-to-win-in-dramatic-australian-gp-opener-amid.5rKGxQHJoDl1P8stMSHlIN) |
| 927 | Belgica 2015 | 301.048 km | Nico Hulkenberg parou no grid; a largada foi abortada e houve uma volta de formacao extra. | [Honda F1](https://global.honda/en/F1/race/2015/Belgium/) |
| 1014 | Japao 2019 | 301.664 km | O painel exibiu a bandeira quadriculada antes da hora; a classificacao oficial ficou na volta 52, em vez da 53. | [FIA: classificacao final](https://www.fia.com/sites/default/files/decision-document/2019%20Japanese%20Grand%20Prix%20-%20Final%20Race%20Classification.pdf) |
| 725 | Alemanha 2004 | 301.884 km | Um problema no carro de Olivier Panis no grid abortou a primeira largada; a prova foi reduzida de 67 para 66 voltas. | [StatsF1: Alemanha 2004](https://www.statsf1.com/en/2004/allemagne.aspx) |
| 869 | Hungria 2012 | 302.249 km | A largada foi abortada depois de Michael Schumacher parar fora da posicao correta; o grid fez outra volta de formacao. | [FIA: relato](https://www.fia.com/news/mercedes-make-point); [Hungaroring: historia do GP](https://hungaroring.hu/guidebook/) |
| 926 | Hungria 2015 | 302.249 km | Felipe Massa estava fora de posicao no grid; a largada foi abortada e a corrida perdeu uma volta. | [FIA: relato da corrida](https://api.fia.com/news/vettel-wins-incident-packed-hungarian-grand-prix) |
| 732 | Australia 2005 | 302.271 km | Kimi Raikkonen parou no grid; a largada foi abortada e a distancia caiu de 58 para 57 voltas. | [Honda F1: resultados de 2005](https://global.honda/en/F1/race/2005/) |
| 753 | Australia 2006 | 302.271 km | Giancarlo Fisichella parou no grid e a largada foi abortada, acrescentando uma volta de formacao. | [Honda F1: resultados de 2006](https://global.honda/en/F1/race/2006/) |
| 898 | Australia 2014 | 302.271 km | Um problema no carro de Max Chilton levou a uma largada abortada e a outra volta de formacao. | [FIA: relato da corrida](https://www.fia.com/news/rosberg-wins-australia) |
| 936 | Australia 2016 | 302.271 km | Daniil Kvyat teve um problema na volta de formacao; o grid fez uma volta extra e a prova caiu de 58 para 57 voltas. | [Honda F1: relato da corrida](https://global.honda/en/F1/race/2016/Australia/) |
| 957 | Australia 2017 | 302.271 km | A direcao de prova abortou a largada por incerteza sobre um sinal amarelo e a situacao no grid; houve volta de formacao extra. | [RaceFans: explicacao da largada abortada](https://www.racefans.net/2017/03/26/reasons-aborted-first-start-explained/) |
| 772 | Espanha 2007 | 302.449 km | Jarno Trulli parou no grid; a largada foi abortada e a prova caiu de 66 para 65 voltas. | [Autosport: revisao da corrida](https://www.autosport.com/general/news/the-complete-2007-spanish-gp-review-5078273/5078273) |
| 1136 | Austria 2025 | 302.692 km | Carlos Sainz nao conseguiu sair para a volta de formacao; a largada foi abortada e a prova caiu de 71 para 70 voltas. | [F1: largada abortada](https://www.formula1.com/en/video/2025-austrian-grand-prix-race-start-aborted-after-sainzs-brakes-catch-fire.1836271745554555069) |
| 1036 | Bahrein 2021 | 302.826 km | Sergio Perez perdeu potencia na volta de formacao; a prova foi reduzida de 57 para 56 voltas. | [F1: incidente na volta de formacao](https://www.formula1.com/en/video/2021-bahrain-grand-prix-sergio-perez-loses-engine-power-on-formation-lap.1696505108064313455) |
| 777 | Gra-Bretanha 2007 | 303.216 km | Felipe Massa parou no grid, causando uma largada abortada e uma volta de formacao extra. | [StatsF1: Gra-Bretanha 2007](https://www.statsf1.com/en/2007/grande-bretagne.aspx) |
| 738 | Europa 2005 | 303.732 km | Giancarlo Fisichella parou no grid; a largada foi abortada e uma volta de formacao extra reduziu a prova de 60 para 59 voltas. | [Honda F1: resultados de 2005](https://global.honda/en/F1/race/2005/) |
| 911 | Singapura 2014 | 303.763 km | A prova chegou ao limite de duas horas e terminou apos 60 das 61 voltas previstas. | [F1: relato da corrida](https://www.formula1.com/en/latest/article/race-hamilton-defeats-vettel-as-rosberg-retires-in-singapore.2GVyV5omVy446CTHhOYxsE.2GVyV5omVy446CTHhOYxsE) |
| 960 | Russia 2017 | 303.897 km | Fernando Alonso parou na volta de formacao; a largada foi abortada e a prova caiu de 53 para 52 voltas. | [Honda F1: resultados da Russia 2017](https://global.honda/en/F1/race/2017/Russian/) |

`dataset_limpo.csv` fica preservado. A saida filtrada e salva separadamente em `dataset_limpo_sem_corridas_encurtadas.csv` para manter rastreavel a diferenca entre a base limpa original e o conjunto preparado para este modelo.


In [ ]:
RACE_IDS_REMOVIDOS = [700, 805, 912, 1047, 1075, 1064, 703, 872, 901, 966, 970, 983, 1008, 1074, 1093, 1126, 927, 1014, 725, 869, 926, 732, 753, 898, 936, 957, 772, 1136, 1036, 777, 738, 911, 960]
CAMINHO_SAIDA = Path('dataset_limpo_sem_corridas_encurtadas.csv')

ids_encontrados = set(df.loc[df['race_id'].isin(RACE_IDS_REMOVIDOS), 'race_id'])
ids_ausentes = set(RACE_IDS_REMOVIDOS) - ids_encontrados
if ids_ausentes:
    raise ValueError(f'race_id nao encontrado no dataset: {sorted(ids_ausentes)}')

linhas_antes = len(df)
corridas_antes = df['race_id'].nunique()
df = df.loc[~df['race_id'].isin(RACE_IDS_REMOVIDOS)].copy()
df.to_csv(CAMINHO_SAIDA, index=False)

print(f'Corridas removidas: {len(RACE_IDS_REMOVIDOS)}')
print(f'Linhas removidas: {linhas_antes - len(df):,}')
print(f'Corridas restantes: {df["race_id"].nunique():,} de {corridas_antes:,}')
print(f'Linhas restantes: {len(df):,}')
print(f'Novo dataset: {CAMINHO_SAIDA}')

In [ ]:
# Recalcula a distribuicao depois de excluir os eventos selecionados.
df_corridas_depois = df_corridas.loc[
    ~df_corridas['race_id'].isin(RACE_IDS_REMOVIDOS)
].copy()
valores_depois = df_corridas_depois['race_distance'].dropna()
q1_depois, q3_depois = valores_depois.quantile([0.25, 0.75])
iqr_depois = q3_depois - q1_depois
limite_inferior_depois = q1_depois - 1.5 * iqr_depois
limite_superior_depois = q3_depois + 1.5 * iqr_depois
fora_iqr_depois = valores_depois.lt(limite_inferior_depois) | valores_depois.gt(limite_superior_depois)

print(f'Corridas apos a remocao: {len(valores_depois):,}')
print(f'Menor distancia restante: {valores_depois.min():.3f} km')
print(f'Limites IQR recalculados: {limite_inferior_depois:.3f} a {limite_superior_depois:.3f} km')
print(f'Corridas ainda sinalizadas pelo IQR: {int(fora_iqr_depois.sum())}')
print('Corridas fora dos limites IQR apos a remocao:')
colunas_tabela_depois = [
    'race_id', 'race_year', 'race_round', 'gp_id', 'circuit_name',
    'race_distance', 'race_laps'
]
df_corridas_depois['fora_do_iqr_depois'] = False
df_corridas_depois.loc[fora_iqr_depois.index, 'fora_do_iqr_depois'] = fora_iqr_depois
display(
    df_corridas_depois.loc[
        df_corridas_depois['fora_do_iqr_depois'], colunas_tabela_depois
    ].sort_values('race_distance')
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), gridspec_kw={'width_ratios': [2, 1]})
axes[0].hist(valores_depois, bins=25, color='#4C9F70', edgecolor='white')
axes[0].axvline(limite_inferior_depois, color='#D9534F', linestyle='--', label='Limite IQR apos remocao')
axes[0].axvline(limite_superior_depois, color='#D9534F', linestyle='--')
axes[0].set(title='Depois da remocao (eixo ajustado)', xlabel='Distancia (km)', ylabel='Numero de corridas', xlim=(valores_depois.min() - 5, valores_depois.max() + 5))
axes[0].legend()
axes[1].boxplot(valores_depois, orientation='horizontal', tick_labels=['Corridas'], patch_artist=True,
                boxprops={'facecolor': '#A8D5B5'})
axes[1].axvline(limite_inferior_depois, color='#D9534F', linestyle='--')
axes[1].axvline(limite_superior_depois, color='#D9534F', linestyle='--')
axes[1].set(title='Boxplot IQR - depois (eixo ajustado)', xlabel='Distancia (km)', xlim=(valores_depois.min() - 5, valores_depois.max() + 5))
plt.tight_layout()
plt.show()

## Decisao nesta etapa

Foram removidos por inteiro os `race_id` 700, 805, 912, 1047, 1075, 1064, 703, 872, 901, 966, 970, 983, 1008, 1074, 1093, 1126, 927, 1014, 725, 869, 926, 732, 753, 898, 936, 957, 772, 1136, 1036, 777, 738, 911 e 960: 33 corridas encurtadas por condicoes excepcionais ou classificadas antes da distancia prevista. Monaco foi excluido das tabelas e dos graficos para nao misturar sua distancia naturalmente menor com a distribuicao geral; os GPs regulares de Monaco continuam no dataset filtrado.

O dataset filtrado foi salvo em `dataset_limpo_sem_corridas_encurtadas.csv`; `dataset_limpo.csv` permanece intacto. O IQR sinaliza casos para revisao, mas nao e usado como regra automatica de remocao.
